# E6 analysis consumer — QuIC vs spectrum within fixed-degree-sequence strata

E6's producer built four strata of 400 connected graphs, each stratum a
fixed nonregular degree sequence, embedded with the flat-encoder canonical
QuIC circuit. This consumer runs the E2S column set within each stratum on
frozen splits: eigenvalues, adjacency trace moments tr(A^k) for k = 3–8,
the QuIC vector, and the concatenation of QuIC with the eigenvalue block
RMS-rescaled per train fold. ΔR² = concat − best spectral column, per row.
All analysis is within-stratum; nothing is compared across strata.

**Pre-registered expectations, written before any result.**
- **Consistency rows, not findings:** the moment column must sit at ~1.000
  on C3 (tr(A³) = 6·C3 holds on any graph) and on C4 (tr(A⁴) = 8·C4 +
  2Σd² − Σd, and Σd² is a stratum constant); the eigenvalue column must
  sit at ~1.000 on spectral_gap (a function of the spectrum). A failure on
  any of these rows means an upstream bug, not a finding.
- **The live row is C5.** Its cubic identity, tr(A⁵) = 10·C5 + 10·tr(A³),
  requires regularity and does not survive off it: the degree-weighted
  triangle term Σ_{triangles} (d_u + d_v + d_w) varies within a stratum.
  The paper sentence E6 exists to earn is whether QuIC beats the direct
  spectral baseline on C5 once the spectrum loses its identity.
- **Secondary live rows:** C6, diameter, radius, wiener.
- **Interpretation guards, pre-registered:** if the spectral columns land
  near 1.000 on C5 anyway (an identity is sufficient for pinning, not
  necessary), the finding is "no residue needed on these strata" and is
  reported as such; if QuIC trails the spectrum on any live row, the row
  is reported as-is. Girth may be constant within a stratum (the producer
  saw sd = 0 on S3 and S4); constant targets are reported as degenerate
  and skipped, and near-constant targets use fold-wise guards rather than
  silent NaN propagation.

**Protocol.** The E2 probe verbatim: RidgeCV(alphas = logspace(-14, 2, 17),
cv = 5) per fold on KFold(5, shuffle, seed 0) splits frozen per stratum and
saved with the results. Low-dimensional spectral blocks are standardized
inside the fold (trace moments span many orders); the QuIC block goes in
raw, as everywhere. The concat column rescales the eigenvalue block per
train fold so its RMS matches the QuIC block's, then concatenates.

**Integrity gates before any probe runs:** stratum counts and degree
sequences; distinct canonical graph6 strings; vector sortedness and
normalization; the two trace identities re-asserted on every graph; the
meta record must say flat encoding; and one graph per stratum is
recomputed through an independent pure-numpy flat circuit (the E2C engine
with a flat encoder) and hard-asserted against the stored vector at 1e-12
— this catches launching the consumer against a stale degree-encoded pkl.

**Runtime.** The QuIC and concat ridges dominate: roughly 430 fits of
320×16384 per target-stratum, a few minutes each, so the full grid is
about 2–4 hours. Spectral columns are seconds. RUN_STRATA is the session
knob for parallel Kaggle launches; results checkpoint per stratum.

## Environment

In [1]:
import pickle

import numpy as np

from itertools import combinations

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from tqdm.notebook import tqdm

In [2]:
SEED = 0
ALPHAS = np.logspace(-14, 2, 17)
N_SPLITS = 5
BASE = "/kaggle/input/notebooks/lukemiller1987/aaai-n14-e6-dataset-producer/"

N_VERTICES = 14
EXPECTED_STRATA = ('S1_near_regular', 'S2_bimodal', 'S3_skewed', 'S4_hub')
EXPECTED_PER_STRATUM = 400
TARGETS = ('C3', 'C4', 'C5', 'C6', 'girth', 'diameter',
           'radius', 'wiener', 'spectral_gap')
MOMENT_KS = (3, 4, 5, 6, 7, 8)
GATE_TOL = 1e-12

# Session knob: subset of strata for parallel Kaggle launches.
RUN_STRATA = EXPECTED_STRATA

## Load and gate

Every integrity property the probes depend on is asserted before anything
runs. The flat-circuit spot check recomputes one stored vector per stratum
through an independent pure-numpy implementation (the E2C engine with the
flat encoder) and fails the whole notebook if the pkl was produced by any
other circuit.

In [3]:
with open(BASE + "n14_e6_data_dict.pkl", 'rb') as f:
    E6 = pickle.load(f)
DATA, META = E6['data'], E6['meta']

assert tuple(sorted(DATA)) == tuple(sorted(EXPECTED_STRATA))
assert META['circuit']['encoding'].startswith('flat'), (
    'pkl is not the flat-encoder run — wrong producer version')
FLAT_ANGLE = META['circuit']['flat_encoding_angle']
TH_ENT = META['circuit']['entangler_scalar']
TH_MIX = META['circuit']['mixer_scalar']

STRATA = {}
for s in EXPECTED_STRATA:
    keys = sorted(DATA[s])
    assert len(keys) == EXPECTED_PER_STRATUM, (
        f'{s}: expected {EXPECTED_PER_STRATUM}, got {len(keys)}')
    seq = sorted(META['sequences'][s])
    g6s = [DATA[s][k]['graph6'] for k in keys]
    assert len(set(g6s)) == len(g6s), f'{s}: duplicate canonical graph6'

    A = np.stack([DATA[s][k]['adj_mat'] for k in keys]).astype(np.int64)
    V = np.vstack([DATA[s][k]['exact_vector'] for k in keys])
    assert V.shape == (len(keys), 1 << N_VERTICES)
    assert np.all(np.diff(V, axis=1) <= 0), f'{s}: vectors not sorted descending'
    assert np.abs(V.sum(axis=1) - 1.0).max() < 1e-12, f'{s}: vectors not normalized'

    d = A.sum(axis=2)
    assert all(sorted(row.tolist()) == seq for row in d), f'{s}: degree sequence broken'
    # trace identities, per graph
    c3 = np.array([DATA[s][k]['C3'] for k in keys])
    c4 = np.array([DATA[s][k]['C4'] for k in keys])
    t3 = np.trace(A @ A @ A, axis1=1, axis2=2)
    t4 = np.trace(np.stack([np.linalg.matrix_power(a, 4) for a in A]),
                  axis1=1, axis2=2)
    assert np.array_equal(t3, 6 * c3), f'{s}: tr(A^3) identity broken'
    assert np.array_equal(t4, 8 * c4 + 2 * (d ** 2).sum(axis=1) - d.sum(axis=1)), (
        f'{s}: tr(A^4) identity broken')

    STRATA[s] = {
        'keys': keys, 'A': A, 'V': V,
        'spectra': np.vstack([DATA[s][k]['spectrum'] for k in keys]),
        'targets': {t: np.array([DATA[s][k][t] for k in keys], dtype=float)
                    for t in TARGETS},
    }
    print(f'{s}: {len(keys)} graphs, identities exact, vectors verified')

S1_near_regular: 400 graphs, identities exact, vectors verified
S2_bimodal: 400 graphs, identities exact, vectors verified
S3_skewed: 400 graphs, identities exact, vectors verified
S4_hub: 400 graphs, identities exact, vectors verified


In [4]:
def quic_probs_flat(adj):
    """Independent pure-numpy flat-encoder QuIC (E2C engine, flat angles)."""
    adj = np.asarray(adj, dtype=np.float64)
    n = adj.shape[0]
    N = 1 << n
    psi = np.ones(1, dtype=np.complex128)
    half = np.float64(FLAT_ANGLE) / 2
    c, s = np.cos(half), np.sin(half)
    for q in range(n):
        psi = np.concatenate([psi * c, psi * (-1j * s)])
    eu, ev = np.nonzero(np.triu(adj, k=1))
    E = len(eu)
    idx = np.arange(N, dtype=np.int64)
    m = np.zeros(N, dtype=np.int64)
    for u, v in zip(eu, ev):
        m += ((idx >> u) & 1) ^ ((idx >> v) & 1)
    ang = -(np.float64(TH_ENT) / 2) * (E - 2 * np.arange(E + 1)).astype(np.float64)
    psi = psi * (np.cos(ang) + 1j * np.sin(ang))[m]
    half = np.float64(TH_MIX) / 2
    c, s = np.cos(half), np.sin(half)
    for q in range(n):
        psi = psi.reshape(-1, 2, 1 << q)
        a0 = psi[:, 0, :].copy()
        a1 = psi[:, 1, :].copy()
        psi[:, 0, :] = c * a0 + (-1j * s) * a1
        psi[:, 1, :] = (-1j * s) * a0 + c * a1
        psi = psi.reshape(-1)
    return np.sort(psi.real ** 2 + psi.imag ** 2)[::-1]

for s in EXPECTED_STRATA:
    v = quic_probs_flat(STRATA[s]['A'][0])
    dmax = np.abs(v - STRATA[s]['V'][0]).max()
    assert dmax < GATE_TOL, f'{s}: flat-circuit spot check FAILED at {dmax:.2e}'
    print(f'{s}: flat-circuit spot check passed ({dmax:.2e})')

S1_near_regular: flat-circuit spot check passed (1.33e-15)
S2_bimodal: flat-circuit spot check passed (4.51e-17)
S3_skewed: flat-circuit spot check passed (3.33e-16)
S4_hub: flat-circuit spot check passed (7.77e-16)


## Frozen splits and probe machinery

KFold(5, shuffle, seed 0) per stratum, frozen here and saved with the
results. The ridge probe is the E2 protocol verbatim. The fold-wise
variance guard handles near-constant targets: a fold whose test targets
are constant records NaN and is counted, never silently averaged.

In [5]:
SPLITS = {}
for s in EXPECTED_STRATA:
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    SPLITS[s] = list(kf.split(np.arange(EXPECTED_PER_STRATUM)))


def ridge_probe(X, y, splits, standardize=False):
    """Per-fold RidgeCV (alpha via nested cv=5), R2 on test; E2 protocol."""
    scores, chosen = [], []
    for tr, te in splits:
        if y[te].std() == 0 or y[tr].std() == 0:
            scores.append(np.nan)
            chosen.append(np.nan)
            continue
        if standardize:
            model = make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS, cv=5))
        else:
            model = RidgeCV(alphas=ALPHAS, cv=5)
        model.fit(X[tr], y[tr])
        scores.append(r2_score(y[te], model.predict(X[te])))
        chosen.append(model[-1].alpha_ if standardize else model.alpha_)
    return np.array(scores), chosen


def concat_probe(Xq, Xs, y, splits):
    """QuIC raw + spectral block rescaled per train fold to match QuIC RMS."""
    scores, chosen = [], []
    for tr, te in splits:
        if y[te].std() == 0 or y[tr].std() == 0:
            scores.append(np.nan)
            chosen.append(np.nan)
            continue
        rms_q = np.sqrt((Xq[tr] ** 2).mean())
        rms_s = np.sqrt((Xs[tr] ** 2).mean())
        scale = rms_q / rms_s
        Xtr = np.hstack([Xq[tr], Xs[tr] * scale])
        Xte = np.hstack([Xq[te], Xs[te] * scale])
        model = RidgeCV(alphas=ALPHAS, cv=5)
        model.fit(Xtr, y[tr])
        scores.append(r2_score(y[te], model.predict(Xte)))
        chosen.append(model.alpha_)
    return np.array(scores), chosen

## Main grid

Per stratum: build the four feature blocks, run every non-degenerate
target through every column, checkpoint the stratum to the results pkl.
The moment block is tr(A^k) for k = 3–8 (k = 1 is zero and k = 2 is the
stratum constant 2m). Degenerate targets (constant within the stratum) are
recorded and skipped.

In [6]:
RESULTS = {}
for s in RUN_STRATA:
    A, V, spectra = STRATA[s]['A'], STRATA[s]['V'], STRATA[s]['spectra']
    B = len(A)
    P = A.astype(np.float64)
    moments = np.empty((B, len(MOMENT_KS)))
    Pk = np.stack([np.linalg.matrix_power(a, MOMENT_KS[0]) for a in A])
    moments[:, 0] = np.trace(Pk, axis1=1, axis2=2)
    for c, k in enumerate(MOMENT_KS[1:], start=1):
        Pk = Pk @ A
        moments[:, c] = np.trace(Pk, axis1=1, axis2=2)

    COLUMNS = {
        'eig':     lambda y: ridge_probe(spectra, y, SPLITS[s], standardize=True),
        'moments': lambda y: ridge_probe(moments, y, SPLITS[s], standardize=True),
        'quic':    lambda y: ridge_probe(V, y, SPLITS[s]),
        'concat':  lambda y: concat_probe(V, spectra, y, SPLITS[s]),
    }

    RESULTS[s] = {}
    for t in tqdm(TARGETS, desc=f'{s} targets'):
        y = STRATA[s]['targets'][t]
        if np.unique(y).size < 2:
            RESULTS[s][t] = {'degenerate': True}
            print(f'{s} {t}: DEGENERATE (constant within stratum) — skipped')
            continue
        row = {'degenerate': False}
        for col, fn in COLUMNS.items():
            sc, al = fn(y)
            row[col] = {'r2_folds': sc, 'r2_mean': float(np.nanmean(sc)),
                        'r2_sd': float(np.nanstd(sc)),
                        'n_nan_folds': int(np.isnan(sc).sum()),
                        'alphas_chosen': al}
        best_spec = float(np.nanmax([row['eig']['r2_mean'],
                                     row['moments']['r2_mean']]))
        row['best_spectral'] = best_spec
        row['delta_concat'] = row['concat']['r2_mean'] - best_spec
        row['delta_quic'] = row['quic']['r2_mean'] - best_spec
        RESULTS[s][t] = row

    with open('/kaggle/working/n14_e6_analysis.pkl', 'wb') as f:
        pickle.dump({'results': RESULTS,
                     'splits': {k: [(tr.tolist(), te.tolist()) for tr, te in v]
                                for k, v in SPLITS.items()},
                     'seed': SEED, 'alphas_grid': ALPHAS,
                     'moment_ks': MOMENT_KS, 'run_strata': RUN_STRATA}, f)
    print(f'{s} complete — checkpointed to n14_e6_analysis.pkl')

S1_near_regular targets:   0%|          | 0/9 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.55438e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=8.87324e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.12851e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=7.05247e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=7.48726e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/pytho

S1_near_regular complete — checkpointed to n14_e6_analysis.pkl


S2_bimodal targets:   0%|          | 0/9 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.41633e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.77889e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=9.31173e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.95396e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=8.09393e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/pytho

...

/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.05981e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.23549e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=6.90297e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.99446e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=8.35023e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/pytho

S3_skewed complete — checkpointed to n14_e6_analysis.pkl


S4_hub targets:   0%|          | 0/9 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=7.02643e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=9.16442e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=8.81817e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=9.45479e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.77468e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/pytho

S4_hub complete — checkpointed to n14_e6_analysis.pkl


## Tables

Per-stratum table, then the equal-weight aggregate over strata (mean of
per-stratum means, non-degenerate rows only). The consistency rows print
with their expected values so a violation is visible at a glance.

In [7]:
CONSISTENCY = {'C3': 'moments ~1.000 (identity)',
               'C4': 'moments ~1.000 (identity + stratum-constant sum d^2)',
               'spectral_gap': 'eig ~1.000 (function of spectrum)'}

for s in RUN_STRATA:
    print(f'\n=== {s} ===')
    print(f'{"target":>13} | {"eig":>14} {"moments":>14} {"quic":>14} '
          f'{"concat":>14} | {"dQuIC":>7} {"dConcat":>8}')
    for t in TARGETS:
        r = RESULTS[s][t]
        if r['degenerate']:
            print(f'{t:>13} | degenerate (constant within stratum)')
            continue
        cells = '  '.join(f"{r[c]['r2_mean']:+.3f}±{r[c]['r2_sd']:.3f}"
                          for c in ('eig', 'moments', 'quic', 'concat'))
        note = f"   <- expect {CONSISTENCY[t]}" if t in CONSISTENCY else ''
        print(f"{t:>13} | {cells} | {r['delta_quic']:+.3f}  "
              f"{r['delta_concat']:+.3f}{note}")

print('\n=== aggregate (equal-weight mean over strata, non-degenerate rows) ===')
print(f'{"target":>13} | {"eig":>7} {"moments":>8} {"quic":>7} {"concat":>7} '
      f'| {"dQuIC":>7} {"dConcat":>8} | {"strata":>6}')
for t in TARGETS:
    rows = [RESULTS[s][t] for s in RUN_STRATA if not RESULTS[s][t]['degenerate']]
    if not rows:
        print(f'{t:>13} | degenerate in every stratum')
        continue
    agg = {c: float(np.nanmean([r[c]['r2_mean'] for r in rows]))
           for c in ('eig', 'moments', 'quic', 'concat')}
    dq = float(np.nanmean([r['delta_quic'] for r in rows]))
    dc = float(np.nanmean([r['delta_concat'] for r in rows]))
    n_valid = sum(1 for r in rows if not np.isnan(r['quic']['r2_mean']))
    print(f"{t:>13} | {agg['eig']:+.3f}  {agg['moments']:+.3f}  "
          f"{agg['quic']:+.3f}  {agg['concat']:+.3f} | {dq:+.3f}  {dc:+.3f} "
          f"| {n_valid}/{len(RUN_STRATA)}")


=== S1_near_regular ===
       target |            eig        moments           quic         concat |   dQuIC  dConcat
           C3 | +0.990±0.002  +1.000±0.000  +0.449±0.714  +0.989±0.002 | -0.551  -0.011   <- expect moments ~1.000 (identity)
           C4 | +0.968±0.006  +1.000±0.000  +0.095±0.048  +0.967±0.006 | -0.905  -0.033   <- expect moments ~1.000 (identity + stratum-constant sum d^2)
           C5 | +0.723±0.050  +0.802±0.037  -0.017±0.056  +0.863±0.045 | -0.819  +0.061
           C6 | +0.669±0.052  +0.757±0.015  -0.021±0.012  +0.661±0.058 | -0.778  -0.096
        girth | +0.468±0.048  +0.343±0.072  +0.007±0.228  +0.436±0.063 | -0.461  -0.032
     diameter | +0.372±0.069  +0.369±0.089  -0.093±0.122  +0.257±0.146 | -0.465  -0.115
       radius | -0.005±0.026  +0.028±0.044  -0.064±0.077  -0.033±0.018 | -0.092  -0.061
       wiener | +0.790±0.051  +0.841±0.025  +0.184±0.147  +0.754±0.083 | -0.657  -0.087
 spectral_gap | +1.000±0.000  +0.891±0.009  +0.389±0.061  +1.000±0.000 | 

## Results

(Written after the run. The reading order: (1) the consistency rows — C3
and C4 at ~1.000 in the moment column and spectral_gap at ~1.000 in the
eigenvalue column certify the pipeline; anything else there is an upstream
bug, not a finding; (2) the C5 row, the reason E6 exists — the comparison
is QuIC vs the best spectral column with the identity gone, and dQuIC /
dConcat carry the sentence in either direction, per the pre-registered
guards; (3) the secondary live rows C6, diameter, radius, wiener, read the
same way; (4) fold-guard counts and degenerate rows, reported not hidden;
(5) the cross-stratum consistency of whatever pattern appears — four
strata are four semi-independent replications of the same question, and
agreement across them is the robustness claim E6 contributes. Verb
discipline as everywhere: "linearly decodable," never "predicts"; the
spectral baseline "retains/loses its identity," and QuIC "beats the tested
spectral baselines on these strata" at most.)